# AI 3D + Animation Engine — One Colab

Two independent engines in one runtime:

- **3D Engine:** image → TRELLIS.2 → Unreal-safe PBR GLB + manifest
- **Animation Engine:** selected humanoid → MIA rig → ARDY → retarget → validated Unreal ZIP

Run either engine separately. If you run both, the Animation Engine can reuse the 3D Engine GLB without downloading/re-uploading it.

> Fresh GPU runtime is assumed. Long commands stream live and write logs to `/content/engine_logs/`.


In [ ]:
# Shared setup — run once before either engine.
import os, pathlib, shutil, subprocess, time, collections

LOG_DIR = pathlib.Path("/content/engine_logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def run_live(cmd, *, cwd=None, env=None, label="process"):
    """Run a command with merged stdout/stderr streamed live into Colab + a log file."""
    cmd = [str(x) for x in cmd]
    safe = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in label)[:60]
    stamp = time.strftime("%Y%m%d_%H%M%S")
    log_path = LOG_DIR / f"{stamp}_{safe}.log"

    print("\n" + "=" * 78, flush=True)
    print(f"[RUN] {label}", flush=True)
    print("[CMD] " + " ".join(cmd), flush=True)
    print(f"[LOG] {log_path}", flush=True)
    print("=" * 78, flush=True)

    started = time.time()
    tail = collections.deque(maxlen=80)
    with log_path.open("w", encoding="utf-8", errors="replace") as log:
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
            tail.append(line.rstrip("\n"))
        rc = proc.wait()

    elapsed = time.time() - started
    if rc != 0:
        print("\n" + "!" * 78, flush=True)
        print(f"[FAILED] {label} | exit={rc} | elapsed={elapsed/60:.1f} min", flush=True)
        print(f"[FULL LOG] {log_path}", flush=True)
        print("[LAST LOG LINES]", flush=True)
        for line in tail:
            print(line, flush=True)
        print("!" * 78, flush=True)
        raise RuntimeError(
            f"{label} failed with exit code {rc}. "
            f"Read the error printed above or open {log_path}."
        )

    print(f"\n[DONE] {label} | elapsed={elapsed/60:.1f} min", flush=True)
    return log_path

print("[COLAB][1/3] Checking GPU runtime...", flush=True)
if shutil.which("nvidia-smi") is None:
    raise RuntimeError(
        "No NVIDIA GPU runtime detected. In Colab: Runtime → Change runtime type → GPU, "
        "then reconnect and run this cell again."
    )
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
    text=True, capture_output=True, check=True,
).stdout.strip().splitlines()
gpu_name, memory_mib = [x.strip() for x in smi[0].rsplit(",", 1)]
memory_mib = int(memory_mib)
free_gib = shutil.disk_usage("/content").free / 1024**3
print(f"[COLAB][1/3] GPU: {gpu_name} | VRAM: {memory_mib/1024:.1f} GiB | Free disk: {free_gib:.1f} GiB", flush=True)
if memory_mib < 24000:
    raise RuntimeError("Use a >=24 GB GPU for the supported pipeline.")
if free_gib < 35:
    raise RuntimeError("Need at least 35 GiB free disk for one engine.")

print("[COLAB][2/3] Downloading current engine helper scripts from GitHub...", flush=True)
REPO = pathlib.Path("/content/My-works")
if REPO.exists():
    shutil.rmtree(REPO)
run_live(
    ["git", "clone", "--progress", "--depth", "1",
     "https://github.com/Logan17de/My-works.git", str(REPO)],
    label="Clone My-works helpers",
)
ENGINE_ROOT = REPO / "ai-3d-animation-engines"
TOOLS_3D = ENGINE_ROOT / "3d-engine"
TOOLS_ANIM = ENGINE_ROOT / "animation-engine"
print("[COLAB][3/3] ✅ Helpers ready.", flush=True)


---
# 💾 Optional Google Drive build cache

Recommended for fresh GPU runtimes. This caches **sources and compiled/native wheels**, not large model weights.


In [ ]:
USE_DRIVE_BUILD_CACHE = True #@param {type:"boolean"}
DRIVE_CACHE_ROOT = "/content/drive/MyDrive/AI3D_Engine_Cache" #@param {type:"string"}

if USE_DRIVE_BUILD_CACHE:
    from google.colab import drive
    print("[CACHE][1/3] Mounting Google Drive...", flush=True)
    drive.mount("/content/drive", force_remount=False)
    cache_root = pathlib.Path(DRIVE_CACHE_ROOT)
    print("[CACHE][2/3] Creating/checking cache folders...", flush=True)
    for name in ("sources", "wheels", "downloads"):
        (cache_root / name).mkdir(parents=True, exist_ok=True)
    os.environ["ENGINE_CACHE_ROOT"] = str(cache_root)
    source_count = len(list((cache_root / "sources").glob("*.tar.gz")))
    wheel_count = len(list((cache_root / "wheels").rglob("*.whl")))
    print(f"[CACHE][3/3] ✅ Enabled: {cache_root}", flush=True)
    print(f"[CACHE] Existing source snapshots: {source_count}", flush=True)
    print(f"[CACHE] Existing compiled wheels: {wheel_count}", flush=True)
else:
    os.environ.pop("ENGINE_CACHE_ROOT", None)
    print("[CACHE] Disabled for this runtime.", flush=True)


---
# 🎨 3D Engine

Run these cells when creating a 3D object or character. **Objects stop here.**


In [ ]:
# Install TRELLIS.2 only.
installer = TOOLS_3D / "install_3d.sh"
print("Checking TRELLIS installer syntax first...", flush=True)
run_live(["bash", "-n", str(installer)], label="TRELLIS installer syntax check")
run_live(["bash", str(installer)], label="TRELLIS.2 installation")


## 🔐 Hugging Face sign-in + gated-model check + visible downloads

TRELLIS.2 uses gated Hugging Face dependencies. Before running this cell, use the **same Hugging Face account** to request/accept access to:

1. `facebook/dinov3-vitl16-pretrain-lvd1689m`
2. `briaai/RMBG-2.0`

For the token, the safest Colab option is **Secrets (🔑) → add `HF_TOKEN`**. A hidden prompt is used as fallback.

This cell verifies the token and gated access **before** the 4B model load, then explicitly pre-downloads required files so you see per-file byte progress and overall percentages.


In [ ]:
# Authenticate safely and pre-download TRELLIS runtime models with visible progress.
import getpass
from google.colab import userdata

HF_TOKEN = None
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Hugging Face READ token (hidden): " ).strip()

if not HF_TOKEN:
    raise ValueError("HF_TOKEN is required.")

# Never print the token.
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HOME"] = "/content/huggingface"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ.pop("HF_HUB_DISABLE_PROGRESS_BARS", None)

hf_env = os.environ.copy()
hf_env["HF_TOKEN"] = HF_TOKEN
hf_env["HF_HOME"] = "/content/huggingface"
hf_env["HF_XET_HIGH_PERFORMANCE"] = "1"
hf_env["PYTHONUNBUFFERED"] = "1"

print("[HF] Token loaded securely. Verifying account + gated model access, then downloading...", flush=True)
run_live(
    [
        "/opt/conda/bin/conda", "run", "--no-capture-output", "-n", "trellis2",
        "python", str(TOOLS_3D / "prepare_hf_models.py"),
    ],
    cwd="/content/TRELLIS.2",
    env=hf_env,
    label="Hugging Face auth + TRELLIS model downloads",
)
print("[HF] ✅ TRELLIS model cache is ready.", flush=True)


In [ ]:
# Upload one reference image and set the asset parameters.
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one reference image.")
INPUT_IMAGE = f"/content/{next(iter(uploaded))}"

ASSET_NAME = "test_character" #@param {type:"string"}
ASSET_TYPE = "character" #@param ["object", "character", "environment", "other"]
TARGET_AXIS = "height" #@param ["width", "height", "depth", "longest"]
TARGET_SIZE_METERS = 1.75 #@param {type:"number"}
DECIMATION_TARGET = 1000000 #@param {type:"integer"}
TEXTURE_SIZE = 4096 #@param {type:"integer"}

if TARGET_SIZE_METERS <= 0:
    raise ValueError("TARGET_SIZE_METERS must be positive.")
print("3D input:", INPUT_IMAGE)
print(f"Asset: {ASSET_NAME} | type={ASSET_TYPE} | {TARGET_AXIS}={TARGET_SIZE_METERS} m")


In [ ]:
# Generate the Unreal-safe GLB.
import json

if not os.environ.get("HF_TOKEN"):
    raise RuntimeError("HF authentication is missing. Run the Hugging Face sign-in/download cell first.")

OUTPUT_3D_DIR = "/content/trellis_outputs"
cmd = [
    "/opt/conda/bin/conda", "run", "--no-capture-output", "-n", "trellis2",
    "python", str(TOOLS_3D / "run_trellis2.py"),
    "--input", INPUT_IMAGE,
    "--output-dir", OUTPUT_3D_DIR,
    "--name", ASSET_NAME,
    "--asset-type", ASSET_TYPE,
    "--target-axis", TARGET_AXIS,
    "--target-size-m", str(TARGET_SIZE_METERS),
    "--envmap", "/content/TRELLIS.2/assets/hdri/forest.exr",
    "--decimation-target", str(DECIMATION_TARGET),
    "--texture-size", str(TEXTURE_SIZE),
]
generation_env = os.environ.copy()
generation_env["HF_HOME"] = "/content/huggingface"
generation_env["HF_XET_HIGH_PERFORMANCE"] = "1"
run_live(
    cmd,
    cwd="/content/TRELLIS.2",
    env=generation_env,
    label="TRELLIS.2 3D generation",
)

GLB_PATH = f"{OUTPUT_3D_DIR}/{ASSET_NAME}.glb"
ASSET_MANIFEST_PATH = f"{OUTPUT_3D_DIR}/{ASSET_NAME}_manifest.json"
PREVIEW_3D_PATH = f"{OUTPUT_3D_DIR}/{ASSET_NAME}_preview.mp4"

for p in (GLB_PATH, ASSET_MANIFEST_PATH):
    if not pathlib.Path(p).is_file():
        raise RuntimeError(f"Missing expected output: {p}")
geometry = json.loads(pathlib.Path(ASSET_MANIFEST_PATH).read_text())["geometry"]
print(json.dumps(geometry, indent=2))
print("3D Engine output:", GLB_PATH)


In [ ]:
# Preview / optionally download the 3D result.
from IPython.display import Video, display
from google.colab import files

if pathlib.Path(PREVIEW_3D_PATH).is_file():
    display(Video(PREVIEW_3D_PATH, embed=True))
else:
    print("No MP4 preview was produced; the GLB can still be valid.")

DOWNLOAD_3D_NOW = False #@param {type:"boolean"}
if DOWNLOAD_3D_NOW:
    files.download(GLB_PATH)
    files.download(ASSET_MANIFEST_PATH)


---
# 🕺 Animation Engine

Can be run independently. Set `USE_3D_ENGINE_OUTPUT=True` only when you want to use the GLB generated above.


In [ ]:
# Install ARDY + Make-It-Animatable only.
installer = TOOLS_ANIM / "install_animation.sh"
print("Checking Animation installer syntax first...", flush=True)
run_live(["bash", "-n", str(installer)], label="Animation installer syntax check")
run_live(["bash", str(installer)], label="Animation Engine installation")


In [ ]:
# Select character and motion settings.
import getpass
from google.colab import files

USE_3D_ENGINE_OUTPUT = False #@param {type:"boolean"}

if USE_3D_ENGINE_OUTPUT:
    TARGET_CHARACTER = globals().get("GLB_PATH")
    if not TARGET_CHARACTER or not pathlib.Path(TARGET_CHARACTER).is_file():
        raise RuntimeError(
            "No live 3D Engine output exists. Run the 3D Engine first or set "
            "USE_3D_ENGINE_OUTPUT=False and upload a humanoid."
        )
    SOURCE_MANIFEST = globals().get("ASSET_MANIFEST_PATH")
    print("Animation input from 3D Engine:", TARGET_CHARACTER)
else:
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one humanoid GLB/FBX/OBJ/PLY.")
    TARGET_CHARACTER = f"/content/{next(iter(uploaded))}"
    SOURCE_MANIFEST = None
    print("Animation input uploaded:", TARGET_CHARACTER)

if pathlib.Path(TARGET_CHARACTER).suffix.lower() not in {".glb", ".fbx", ".obj", ".ply"}:
    raise ValueError("Unsupported humanoid format.")

# Reuse the HF token from the 3D section if available; otherwise read Colab Secret / hidden prompt.
if not os.environ.get("HF_TOKEN"):
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
    if not HF_TOKEN:
        HF_TOKEN = getpass.getpass("Hugging Face READ token (hidden; ARDY/Llama access): " ).strip()
    if not HF_TOKEN:
        raise ValueError("HF token required.")
    os.environ["HF_TOKEN"] = HF_TOKEN

PROMPT = "A person walks forward, stops, and waves with the right hand." #@param {type:"string"}
DURATION_SECONDS = 6.0 #@param {type:"number"}
SEED = 0 #@param {type:"integer"}
TARGET_ALREADY_RIGGED = False #@param {type:"boolean"}
MIA_NO_FINGERS = True #@param {type:"boolean"}


In [ ]:
# Run the complete Animation Engine.
OUTPUT_ANIM_DIR = "/content/animation_outputs"
cmd = [
    "python", str(TOOLS_ANIM / "run_animation_pipeline.py"),
    "--character", TARGET_CHARACTER,
    "--prompt", PROMPT,
    "--duration", str(DURATION_SECONDS),
    "--seed", str(SEED),
    "--output-dir", OUTPUT_ANIM_DIR,
]
if SOURCE_MANIFEST and pathlib.Path(SOURCE_MANIFEST).is_file():
    cmd += ["--source-manifest", SOURCE_MANIFEST]
if TARGET_ALREADY_RIGGED:
    cmd.append("--already-rigged")
if MIA_NO_FINGERS:
    cmd.append("--no-fingers")

run_live(cmd, env=os.environ.copy(), label="Complete Animation Engine pipeline")

MOTION_PREVIEW = f"{OUTPUT_ANIM_DIR}/motion_preview.mp4"
FINAL_FBX = f"{OUTPUT_ANIM_DIR}/character_animated.fbx"
CONTRACT_REPORT = f"{OUTPUT_ANIM_DIR}/animation_contract_report.json"
PACKAGE_ZIP = f"{OUTPUT_ANIM_DIR}/unreal_character_package.zip"

for p in (MOTION_PREVIEW, FINAL_FBX, CONTRACT_REPORT, PACKAGE_ZIP):
    if not pathlib.Path(p).is_file():
        raise RuntimeError(f"Missing animation output: {p}")
print("Animation Engine complete:", FINAL_FBX)


In [ ]:
# Preview motion and download the Unreal package.
from IPython.display import Video, display
from google.colab import files

display(Video(MOTION_PREVIEW, embed=True))
files.download(PACKAGE_ZIP)


---
## Fresh GPU workflow

```text
Fresh GPU runtime
    ↓
Shared Setup
    ↓
(optional) Mount Drive build cache
    ↓
Install only the engine you need
    ↓
For 3D: HF sign-in → gated-access checks → visible model downloads
    ↓
Generate
    ↓
Download outputs
    ↓
Disconnect GPU
```

Model weights still download fresh to local Colab storage. Sources and compatible compiled wheels can be restored from Drive.
